# Entrenamiento en Google Colab

Entrenamiento en dos niveles (triaje binario y especialista multiclase) con la cascada de calibración clínica. La primera celda prepara el entorno: clona el código del repositorio e instala las dependencias, y carga el dataset y los checkpoints desde Google Drive. Requiere una GPU.

In [ ]:
# ============================================================
# Preparación del entorno en Google Colab
# ============================================================
# 1) Clona el código del proyecto (src/, scripts/, requirements.txt).
# 2) Instala las dependencias.
# 3) Monta Google Drive (datos y checkpoints).
# 4) Descomprime el dataset (imágenes + CSVs) al disco local de Colab.
# 5) Enlaza data/ (lectura) y models/ (escritura en Drive).
#
# Requisitos previos en tu Google Drive (MyDrive/SKIN-PROYECT/):
#   - skin_data_cloud.zip   (dataset: contiene data/raw y data/processed)
#   - models/               (checkpoints; se crea solo si no existe)

import os
import sys
import glob
import subprocess

REPO_URL = "https://github.com/Super1232/SKIN-MOTOR-URB.git"  # URL de tu repositorio
PROJECT_ROOT = "/content/SKIN-PROYECT"
DRIVE_ROOT = "/content/drive/MyDrive/SKIN-PROYECT"

# 1. Código del proyecto
if not os.path.exists(os.path.join(PROJECT_ROOT, "src")):
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
print(f"Código del proyecto en: {PROJECT_ROOT}")

# 2. Dependencias
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 3. Google Drive
from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=False)
    print("Google Drive montado en /content/drive")
except Exception as e:
    raise RuntimeError(
        f"No se pudo montar Google Drive: {e}\n"
        "Verifica: 1) aceptar el prompt de autorización, 2) conexión de red, "
        "3) si el runtime se reinició, reinicia el kernel y ejecuta de nuevo."
    ) from e

# 4. Dataset: descomprimir el zip al disco local de Colab (imágenes + CSVs)
zip_candidates = [
    f"{DRIVE_ROOT}/skin_data_cloud.zip",
    "/content/drive/MyDrive/skin_data_cloud.zip",
]
ZIP_PATH = next((p for p in zip_candidates if os.path.exists(p)), None)
if ZIP_PATH is None:
    raise FileNotFoundError(
        "No se encuentra 'skin_data_cloud.zip' en tu Drive "
        f"(buscado en: {zip_candidates})."
    )
if not os.path.exists("/content/data/raw"):
    print("Descomprimiendo el dataset al disco local (puede tardar unos minutos)...")
    subprocess.check_call(["unzip", "-q", "-n", ZIP_PATH, "-d", "/content"])

# 5. Enlaces: data/ (raw + processed) y models/ (en Drive)
os.makedirs("data", exist_ok=True)
for sub in ("raw", "processed"):
    origen = f"/content/data/{sub}"
    destino = f"data/{sub}"
    if os.path.exists(origen) and not os.path.exists(destino):
        os.symlink(origen, destino)
if not glob.glob("data/raw/*"):
    raise RuntimeError(
        "No se encontraron datasets en data/raw tras descomprimir. "
        "Revisa que skin_data_cloud.zip contenga data/raw/."
    )

os.makedirs(f"{DRIVE_ROOT}/models/checkpoints", exist_ok=True)
if not os.path.exists("models"):
    os.symlink(f"{DRIVE_ROOT}/models", "models")

print("Entorno listo.")

## Instalar Dependencias

In [ ]:
import os
req_path = 'requirements.txt' if os.path.exists('requirements.txt') else '../requirements.txt'
!pip install -q -r {req_path}

## Entrenamiento — Nivel 1 (Triaje binario)

In [ ]:
# =============================================================================
# CONFIGURACIÓN CLÍNICA
# =============================================================================


!mkdir -p /content/drive/MyDrive/SKIN-PROYECT/logs
!set -o pipefail; PYTHONPATH=. python -u src/training/train.py \
    --mode binary --backbone_name effnet_b3 \
    --batch_size 64 --epochs 25 --lr 1e-4 --patience 8 --accum_steps 1 \
    --tta_val_views 4 --use_model_soup --poly_epsilon 1.0 \
    --group_balance_dims site age --group_balance_smooth 0.7 --min_positive 10 \
    --selection_metric pauc_sens80 --pauc_sens_target 0.80 \
    --recall_floor_binary 0.90 --focal_alpha 0.75 --use_hybrid_pool \
    2>&1 | tee "/content/drive/MyDrive/SKIN-PROYECT/logs/train_binary_cnn_$(date +%Y%m%d_%H%M%S).log"


# =============================================================================
# CONFIGURACIÓN ALTERNATIVA — VIT
# =============================================================================
# !mkdir -p /content/drive/MyDrive/SKIN-PROYECT/logs
# !set -o pipefail; PYTHONPATH=. python -u src/training/train.py \
#    --mode binary --backbone_name dinov2_vitl14 \
#    --batch_size 48 --epochs 25 --lr 1e-4 --patience 8 \
#    --accum_steps 1 \
#    --tta_val_views 4 --use_model_soup --poly_epsilon 1.0 \
#    --group_balance_dims site age --group_balance_smooth 0.7 \
#    --min_positive 10 \
#    --selection_metric pauc_sens80 --pauc_sens_target 0.80 \
#    --recall_floor_binary 0.90 \
#    --focal_alpha 0.75 \
#    --use_hybrid_pool \
#    2>&1 | tee "/content/drive/MyDrive/SKIN-PROYECT/logs/train_binary_$(date +%Y%m%d_%H%M%S).log"

## Calibración del modelo binario (Temperature + Per-Site Threshold + Conformal)

Tras el entrenamiento binario, esta celda ejecuta la cascada de calibración
clínica REQUERIDA para que el modelo sea desplegable. Sin estos pasos, el
binario tiene:
- Probabilidades sobreconfiadas (sin Temperature Scaling)
- Threshold global 0.20 que produce spec=0% en palms/soles (sin per-site threshold)
- Sin garantía conformal de cobertura ni control de FNR


In [ ]:
import os
import subprocess
import sys

_drive_ckpt = '/content/drive/MyDrive/SKIN-PROYECT/models/checkpoints'
env = {**os.environ, 'PYTHONPATH': f".:{os.environ.get('PYTHONPATH', '')}"}


def _run(cmd, label, capture=False, fatal=True):
    """Ejecuta subprocess. capture=True serializa stdout+stderr en el .ipynb
    (En Colab la salida de un subproceso con fds heredados NO se
    guarda en el notebook; capturarla hace legible el mensaje real, p.ej. la T
    fuera de rango). fatal=True -> raise si returncode!=0. Devuelve el result."""
    print(f'\n=== {label} ===')
    print(f'>>> {" ".join(cmd)}')
    if capture:
        result = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    else:
        result = subprocess.run(cmd, env=env)
    if result.returncode != 0 and fatal:
        raise RuntimeError(
            f'{label} fallo (return code {result.returncode}). '
            'Revisar logs arriba.'
        )
    return result


_candidates = [
    ('SOUP', f'{_drive_ckpt}/best_binary_model_SOUP.pth'),
    ('best', f'{_drive_ckpt}/best_binary_model.pth'),
    ('EMA',  f'{_drive_ckpt}/best_binary_model_EMA.pth'),
]
_candidates = [(n, p) for n, p in _candidates if os.path.exists(p)]
if not _candidates:
    raise FileNotFoundError(
        f'ERROR: No existe ningun checkpoint binario en {_drive_ckpt}/. '
        'Ejecuta primero la celda de entrenamiento binario.'
    )

binary_ckpt = None
for _name, _cand in _candidates:
    _res = _run(
        [sys.executable, 'scripts/calibrate_temperature.py',
         '--mode', 'binary', '--binary-checkpoint', _cand],
        f'[1/3] Temperature Scaling -- probando {_name} ({os.path.basename(_cand)})',
        capture=True, fatal=False,
    )
    if _res.returncode == 0:
        binary_ckpt = _cand
        print(f'\nTemperature Scaling OK con {_name}: {os.path.basename(_cand)}')
        break
    print(f'\n{_name} no calibra (return code {_res.returncode}). Ver el error '
          'capturado arriba (puede ser reconstruccion o T fuera de rango). '
          'Probando siguiente candidato...')

if binary_ckpt is None:
    raise RuntimeError(
        'NINGUN checkpoint binario [SOUP, best, EMA] pudo calibrarse. Revisar el '
        'error capturado arriba para CADA intento. Si es un mismatch de '
        'load_state_dict -> problema de reconstruccion del modelo. '
        'Si la T queda fuera de [0.3, 5.0] en los 3 candidatos -> mis-calibracion '
        'GENUINA del modelo; revisar el entrenamiento: '
        'interaccion label smoothing medico (0.05/0.95) + --focal_alpha + Model '
        'Soup (comprime los logits). NO relajar el guard [0.3, 5.0] '
        '(rango clinico): recalibrar el MODELO. Ver arriba el valor real de T por intento.'
    )

# Persistir el binario desplegable para la calibración multiclase e inferencia (consistencia:
# el conformal binario que se re-genera en la calibración multiclase usa este mismo temperature_binary.pt).
_deployed_marker = f'{_drive_ckpt}/deployed_binary_checkpoint.txt'
with open(_deployed_marker, 'w') as _f:
    _f.write(binary_ckpt + '\n')

# ---------------------------------------------------------------------------
# Paso 2: Per-Site Threshold -- mismo checkpoint elegido.
# ---------------------------------------------------------------------------
print('\n(per-site + conformal hacen forward sobre val: pueden tardar varios minutos)')
_run(
    [sys.executable, 'scripts/calibrate_per_site_threshold.py',
     '--binary-checkpoint', binary_ckpt,
     '--temperature-binary', f'{_drive_ckpt}/temperature_binary.pt',
     '--target-recall', '0.95',
     '--min-subgroup-n', '30',
     '--min-positive', '10'],
    '[2/3] Per-Site Threshold Calibration (target_recall=0.95)'
)

# ---------------------------------------------------------------------------
# Paso 3: Conformal Prediction + Risk Control -- mismo ckpt.
# ---------------------------------------------------------------------------
_run(
    [sys.executable, 'scripts/calibrate_conformal.py',
     '--binary-checkpoint', binary_ckpt,
     '--temperature-binary', f'{_drive_ckpt}/temperature_binary.pt',
     '--alpha-binary', '0.03',
     '--risk-control-fnr', '0.05'],
    '[3/3] Conformal + Risk Control'
)

print('\n' + '=' * 70)
print(f'Calibracion binaria COMPLETA. Binario DESPLEGABLE: {os.path.basename(binary_ckpt)}')
print('   -> Usa ESTE checkpoint para inferencia y para comparar contra DINOv2.')
print(f'   (registrado en {os.path.basename(_deployed_marker)})')
print('=' * 70)
expected_artifacts = [
    'temperature_binary.pt',
    'per_site_threshold_binary.pt',
    'conformal_binary.pt',
    'conformal_risk_binary.pt',
]
for art in expected_artifacts:
    art_path = f'{_drive_ckpt}/{art}'
    if os.path.exists(art_path):
        size_kb = os.path.getsize(art_path) / 1024
        print(f'{art} ({size_kb:.1f} KB)')
    else:
        import warnings
        warnings.warn(
            f'Archivo esperado NO existe: {art}. '
            'Revisar logs del paso correspondiente.',
            RuntimeWarning,
            stacklevel=2,
        )


## Entrenamiento — Nivel 2 (Especialista multiclase)

In [ ]:

import os
_drive_ckpt = '/content/drive/MyDrive/SKIN-PROYECT/models/checkpoints'
_soup_path = f'{_drive_ckpt}/best_binary_model_SOUP.pth'
_baseline_path = f'{_drive_ckpt}/best_binary_model.pth'
if os.path.exists(_soup_path):
    teacher_path = _soup_path
    print(f'Teacher seleccionado: best_binary_model_SOUP.pth (modelo soup, recomendado)')
elif os.path.exists(_baseline_path):
    teacher_path = _baseline_path
    print(f'Teacher SOUP NO existe. Usando fallback: best_binary_model.pth')
    print(f'    Causa probable: class_health bloqueo el SOUP (melanoma_AUC<0.80 o recall<0.30).')
    print(f'    Revisar fairness_report_binary_soup.json antes de promover este modelo a produccion.')
else:
    raise FileNotFoundError(
        f'ERROR: No existe NINGUN checkpoint binario en {_drive_ckpt}/. '
        'Ejecuta primero la celda de entrenamiento binario.'
    )
print(f'Iniciando training multiclase con teacher KD + hybrid pool...')
!mkdir -p /content/drive/MyDrive/SKIN-PROYECT/logs
!set -o pipefail; PYTHONPATH=. python -u src/training/train.py \
    --mode multiclass --backbone_name effnet_b4 \
    --batch_size 32 --epochs 30 --lr 1e-4 --patience 8 --accum_steps 2 \
    --tta_val_views 4 --use_model_soup --poly_epsilon 1.0 \
    --group_balance_dims site age --group_balance_smooth 0.7 --min_positive 10 \
    --distill_from {teacher_path} \
    --use_fitnet --fitnet_teacher_dim 1536 --use_self_distill --use_hybrid_pool \
    2>&1 | tee "/content/drive/MyDrive/SKIN-PROYECT/logs/train_multiclass_cnn_$(date +%Y%m%d_%H%M%S).log"

# =============================================================================
# A/B VIT
# =============================================================================
# !mkdir -p /content/drive/MyDrive/SKIN-PROYECT/logs
# !set -o pipefail; PYTHONPATH=. python -u src/training/train.py \
#    --mode multiclass --backbone_name dinov2_vitl14 \
#    --batch_size 16 --epochs 30 --lr 1e-4 --patience 8 \
#    --accum_steps 4 \
#    --tta_val_views 4 --use_model_soup --poly_epsilon 1.0 \
#    --group_balance_dims site age --group_balance_smooth 0.7 \
#    --min_positive 10 \
#    --distill_from {teacher_path} \
#    --use_fitnet --fitnet_teacher_dim 1024 \
#    --use_self_distill \
#    --use_hybrid_pool \
#    --grad_checkpointing \
#    2>&1 | tee "/content/drive/MyDrive/SKIN-PROYECT/logs/train_multiclass_$(date +%Y%m%d_%H%M%S).log"
# =============================================================================

## Calibración del modelo multiclase (Vector Scaling + Conformal APS)

Tras el entrenamiento multiclase, esta celda calibra el especialista para
desplegar la cascada clínica completa. Incluye:
- **Vector Scaling** (Guo 2017, Tabla 3): T por clase reduce ECE
  15-30% en clases raras como Dermatofibroma (84 imgs).
- **APS Conformal** (Romano NeurIPS 2020, alpha=0.10 informativo):
  set adaptativo de clases plausibles `["Melanoma", "Nevus"]` para casos ambiguos.

**Coste:** ~5-10 min en A100. **Output:** 2 artefactos en `models/checkpoints/`:
- `temperature_multiclass.pt` (Vector Scaling con num_classes=8)
- `conformal_multiclass.pt` (APS, alpha=0.10)

In [ ]:
# Cascada de calibración multiclase automatizada.
# Tras el entrenamiento multiclase, ejecuta los 2 calibradores en orden:
#   1. Vector Scaling (T por clase, Guo 2017 Tabla 3)
#   2. APS Conformal (alpha=0.10 informativo)
import os
import subprocess
import sys

_drive_ckpt = '/content/drive/MyDrive/SKIN-PROYECT/models/checkpoints'

# Auto-detect MULTICLASE: SOUP preferente, fallback a best_multiclass_model.pth.
_soup_multi = f'{_drive_ckpt}/best_multiclass_model_SOUP.pth'
_baseline_multi = f'{_drive_ckpt}/best_multiclass_model.pth'
if os.path.exists(_soup_multi):
    multi_ckpt = _soup_multi
    print(f'Calibrando multiclase contra: best_multiclass_model_SOUP.pth')
elif os.path.exists(_baseline_multi):
    multi_ckpt = _baseline_multi
    print(f'SOUP multiclase no existe. Calibrando contra fallback best_multiclass_model.pth')
    print(f'    Causa probable: class_health bloqueo el SOUP.')
    print(f'    Revisar fairness_report_multiclass_soup.json antes de promover.')
else:
    raise FileNotFoundError(
        f'ERROR: No existe ningun checkpoint multiclase en {_drive_ckpt}/. '
        'Ejecuta primero la celda de entrenamiento multiclase.'
    )

# Auto-detect BINARIO: usar el MISMO checkpoint que calibró la celda de calibración binaria
# (registrado en deployed_binary_checkpoint.txt). `calibrate_conformal.py` SIEMPRE
# corre el binario (re-genera conformal_binary.pt usando temperature_binary.pt), asi
# que DEBE ser el mismo binario que calibró la celda binaria, o la T no corresponderá a sus
# logits (la calibración binaria puede haber elegido best/EMA si el SOUP no calibraba).
# Si el marcador no existe (calibración binaria no ejecutada en esta sesión), fallback SOUP -> best.
_deployed_marker = f'{_drive_ckpt}/deployed_binary_checkpoint.txt'
binary_ckpt = None
if os.path.exists(_deployed_marker):
    with open(_deployed_marker) as _f:
        _cand = _f.read().strip()
    if _cand and os.path.exists(_cand):
        binary_ckpt = _cand
        print(f'Binario (heredado de la calibración binaria): {os.path.basename(binary_ckpt)}')
if binary_ckpt is None:
    _soup_bin = f'{_drive_ckpt}/best_binary_model_SOUP.pth'
    _baseline_bin = f'{_drive_ckpt}/best_binary_model.pth'
    if os.path.exists(_soup_bin):
        binary_ckpt = _soup_bin
    elif os.path.exists(_baseline_bin):
        binary_ckpt = _baseline_bin
    else:
        raise FileNotFoundError(
            f'ERROR: Binary checkpoint requerido para calibrate_conformal.py no existe. '
            f'Ejecuta primero la celda de entrenamiento binario.'
        )

env = {**os.environ, 'PYTHONPATH': f".:{os.environ.get('PYTHONPATH', '')}"}


def _run(cmd, label, capture=False, fatal=True):
    """Ejecuta subprocess. capture=True serializa stdout+stderr en el .ipynb
    (mismo patrón que la calibración binaria: hace legible un fallo del subproceso.
    fatal=True -> raise si returncode!=0."""
    print(f'\n=== {label} ===')
    print(f'>>> {" ".join(cmd)}')
    if capture:
        result = subprocess.run(cmd, env=env, capture_output=True, text=True)
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    else:
        result = subprocess.run(cmd, env=env)
    if result.returncode != 0 and fatal:
        raise RuntimeError(
            f'{label} fallo (return code {result.returncode}). '
            'Revisar logs arriba.'
        )
    return result

# Paso 1: Vector Scaling -- capture=True para que un fallo de calibración sea legible.
_run(
    [sys.executable, 'scripts/calibrate_temperature.py',
     '--mode', 'multiclass',
     '--multiclass-checkpoint', multi_ckpt],
    '[1/2] Vector Scaling Multiclase (Guo 2017 Tabla 3)',
    capture=True,
)

# Paso 2: APS Conformal (alpha=0.10)
# Nota: calibrate_conformal.py SIEMPRE corre el binario (no opcional). Usa el
# binary_ckpt heredado de la calibración binaria para que conformal_binary.pt quede consistente
# con su temperature_binary.pt. Coste: ~5 min extra (forward sobre val).
_run(
    [sys.executable, 'scripts/calibrate_conformal.py',
     '--binary-checkpoint', binary_ckpt,
     '--multiclass-checkpoint', multi_ckpt,
     '--temperature-binary', f'{_drive_ckpt}/temperature_binary.pt',
     '--temperature-multiclass', f'{_drive_ckpt}/temperature_multiclass.pt',
     '--alpha-binary', '0.03',
     '--alpha-multiclass', '0.10'],
    '[2/2] APS Conformal Multiclase + re-Binary Conformal'
)

print('\nCalibracion multiclase COMPLETA. Artefactos generados:')
expected_artifacts = [
    'temperature_multiclass.pt',
    'conformal_multiclass.pt',
]
for art in expected_artifacts:
    art_path = f'{_drive_ckpt}/{art}'
    if os.path.exists(art_path):
        size_kb = os.path.getsize(art_path) / 1024
        print(f'{art} ({size_kb:.1f} KB)')
    else:
        import warnings
        warnings.warn(
            f'Archivo esperado NO existe: {art}. '
            'Revisar logs del paso correspondiente.',
            RuntimeWarning,
            stacklevel=2,
        )

print('\nCASCADA CLINICA COMPLETA. Artefactos finales en Drive:')
print(f'   {_drive_ckpt}/')
print('   ├── best_binary_model.pth (+ _SWA, _EMA, _SOUP)')
print('   ├── best_multiclass_model.pth (+ _SWA, _EMA, _SOUP)')
print('   ├── temperature_binary.pt + temperature_multiclass.pt')
print('   ├── per_site_threshold_binary.pt')
print('   ├── conformal_binary.pt + conformal_risk_binary.pt')
print('   ├── conformal_multiclass.pt')
print('   └── fairness_report_*.json + selective_prediction_curve_*.json')
